In [2]:
!pip install ipymol plip

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 612.4/612.4 kB 10.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 24.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 22.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 47.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 52.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 48.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 4.9 MB/s  0:00:000m eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 31.6 MB/s  0:00:00m0:00:01:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 54.0 MB/s  0:00:00
  error: subprocess-exited-with-error
  
  × Building wheel for plip (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [101 lines of output]
      /tm

In [1]:
from pdbfixer import PDBFixer
from openmm.app import PDBFile

fixer = PDBFixer(filename='HSA_Target/2BXD/prep_2bxd.pdb')
fixer.findMissingResidues()
fixer.findMissingAtoms()
fixer.addMissingAtoms()
fixer.addMissingHydrogens(pH=7.4)
PDBFile.writeFile(fixer.topology, fixer.positions, open('HSA_Target/2BXD/2bxd_fixed.pdb','w'))
print("Wrote HSA_Target/2BXD/2bxd_fixed.pdb")

fixer = PDBFixer(filename='HSA_Target/2BXG/prep_2bxg.pdb')
fixer.findMissingResidues()
fixer.findMissingAtoms()
fixer.addMissingAtoms()
fixer.addMissingHydrogens(pH=7.4)
PDBFile.writeFile(fixer.topology, fixer.positions, open('HSA_Target/2BXG/2bxg_fixed.pdb','w'))
print("Wrote HSA_Target/2BXG/2bxg_fixed.pdb")

fixer = PDBFixer(filename='HSA_Target/1AO6/prep_1ao6.pdb')
fixer.findMissingResidues()
fixer.findMissingAtoms()
fixer.addMissingAtoms()
fixer.addMissingHydrogens(pH=7.4)
PDBFile.writeFile(fixer.topology, fixer.positions, open('HSA_Target/1AO6/1ao6_fixed.pdb','w'))
print("Wrote HSA_Target/1AO6/1ao6_fixed.pdb")

Wrote HSA_Target/2BXD/2bxd_fixed.pdb
Wrote HSA_Target/2BXG/2bxg_fixed.pdb
Wrote HSA_Target/1AO6/1ao6_fixed.pdb


In [2]:
import os
import subprocess

def plipprep(input_file, output_file):
    lines = []
    with open(input_file, "r") as f:
        for line in f:
            if line.startswith(("HEADER", "TITLE", "REMARK", "END")):
                continue
            lines.append(line.rstrip("\n"))

    atom_counter = 1
    fixed_lines = []
    for line in lines:
        if line.startswith(("ATOM", "HETATM")):
            new_line = f"{line[:6]}{atom_counter:5d}{line[11:]}"
            fixed_lines.append(new_line)
            atom_counter += 1
        else:
            fixed_lines.append(line)

    fixed_lines.append("END")

    with open(output_file, "w") as f:
        f.write("\n".join(fixed_lines) + "\n")


targets = {
    "2BXD": {
        "site": "Site1",
        "path": "HSA_Target/2BXD/2bxd_clean.pdb",
        "ligands": ["ENA", "ENA1", "ENB", "ENB1", "RWF", "IBP"],
        "condition": "HOLO",
    },
    "2BXG": {
        "site": "Site2",
        "path": "HSA_Target/2BXG/2bxg_clean.pdb",
        "ligands": ["ENA", "ENA1", "ENB", "ENB1", "RWF", "IBP"],
        "condition": "HOLO",
    },
}

for pdb_id, info in targets.items():
    protein = info["path"]
    ligands = info["ligands"]
    condition = info["condition"]

    if pdb_id == "1AO6":
        sites = ["Site1", "Site2"]
    else:
        sites = [info["site"]]

    for site in sites:
        for ligand in ligands:
            ligand_dir = f"Results/{site}/{condition}/{ligand}"
            ligand_file = f"{ligand}-{site}_1.pdb"
            out_file = f"{ligand}-{site}_PLIP.pdb"

            ligand_path = os.path.join(ligand_dir, ligand_file)
            concat_tmp = os.path.join(ligand_dir, "tmp_concat.pdb")
            out_path = os.path.join(ligand_dir, out_file)
    
            cmd = f"cat {protein} {ligand_path} > {concat_tmp}"
            subprocess.run(cmd, shell=True, check=True)
            
            plipprep(concat_tmp, out_path)
            os.remove(concat_tmp)

In [3]:
import os
import subprocess

def plipprep(input_file, output_file):
    lines = []
    with open(input_file, "r") as f:
        for line in f:
            if line.startswith(("HEADER", "TITLE", "REMARK", "END")):
                continue
            lines.append(line.rstrip("\n"))

    atom_counter = 1
    fixed_lines = []
    for line in lines:
        if line.startswith(("ATOM", "HETATM")):
            new_line = f"{line[:6]}{atom_counter:5d}{line[11:]}"
            fixed_lines.append(new_line)
            atom_counter += 1
        else:
            fixed_lines.append(line)

    fixed_lines.append("END")

    with open(output_file, "w") as f:
        f.write("\n".join(fixed_lines) + "\n")


targets = {
    "2BXD": {
        "site": "Site1",
        "path": "HSA_Target/2BXD/2bxd_clean.pdb",
        "ligands": ["ENA", "ENA1", "ENB", "ENB1", "RWF", "IBP", "SUC"],
        "condition": "HOLO",
    },
    "2BXG": {
        "site": "Site2",
        "path": "HSA_Target/2BXG/2bxg_clean.pdb",
        "ligands": ["ENA", "ENA1", "ENB", "ENB1", "RWF", "IBP"],
        "condition": "HOLO",
    },
}

for pdb_id, info in targets.items():
    protein = info["path"]
    ligands = info["ligands"]
    condition = info["condition"]

    if pdb_id == "1AO6":
        sites = ["Site1", "Site2"]
    else:
        sites = [info["site"]]

    for site in sites:
        for ligand in ligands:
            ligand_dir = f"Results/{site}/{condition}/{ligand}"
            ligand_file = f"{ligand}-{site}_1.pdb"
            out_file = f"{ligand}-{site}_PLIP.pdb"

            ligand_path = os.path.join(ligand_dir, ligand_file)
            concat_tmp = os.path.join(ligand_dir, "tmp_concat.pdb")
            out_path = os.path.join(ligand_dir, out_file)
    
            cmd = f"cat {protein} {ligand_path} > {concat_tmp}"
            subprocess.run(cmd, shell=True, check=True)
            
            plipprep(concat_tmp, out_path)
            os.remove(concat_tmp)

In [3]:
!plip -f Results/Site1/HOLO/ENA/ENA-Site1_PLIP.pdb -t -x -o Results/PLIP/HOLO --name ENA-Site1
!plip -f Results/Site1/HOLO/ENA1/ENA1-Site1_PLIP.pdb -t -x -o Results/PLIP/HOLO --name ENA1-Site1
!plip -f Results/Site1/HOLO/ENB/ENB-Site1_PLIP.pdb -t -x -o Results/PLIP/HOLO --name ENB-Site1
!plip -f Results/Site1/HOLO/ENB1/ENB1-Site1_PLIP.pdb -t -x -o Results/PLIP/HOLO --name ENB1-Site1
!plip -f Results/Site1/HOLO/RWF/RWF-Site1_PLIP.pdb -t -x -o Results/PLIP/HOLO --name RWF-Site1
!plip -f Results/Site1/HOLO/IBP/IBP-Site1_PLIP.pdb -t -x -o Results/PLIP/HOLO --name IBP-Site1

!plip -f Results/Site2/HOLO/ENA/ENA-Site2_PLIP.pdb -t -x -o Results/PLIP/HOLO --name ENA-Site2
!plip -f Results/Site2/HOLO/ENA1/ENA1-Site2_PLIP.pdb -t -x -o Results/PLIP/HOLO --name ENA1-Site2
!plip -f Results/Site2/HOLO/ENB/ENB-Site2_PLIP.pdb -t -x -o Results/PLIP/HOLO --name ENB-Site2
!plip -f Results/Site2/HOLO/ENB1/ENB1-Site2_PLIP.pdb -t -x -o Results/PLIP/HOLO --name ENB1-Site2
!plip -f Results/Site2/HOLO/RWF/RWF-Site2_PLIP.pdb -t -x -o Results/PLIP/HOLO --name RWF-Site2
!plip -f Results/Site2/HOLO/IBP/IBP-Site2_PLIP.pdb -t -x -o Results/PLIP/HOLO --name IBP-Site2

2025-09-06 02:23:36,680 [INFO] [plipcmd.py:124] plip.plipcmd: Protein-Ligand Interaction Profiler (PLIP) 2.3.1
2025-09-06 02:23:36,680 [INFO] [plipcmd.py:125] plip.plipcmd: brought to you by: PharmAI GmbH (2020-2021) - www.pharm.ai - hello@pharm.ai
2025-09-06 02:23:36,680 [INFO] [plipcmd.py:126] plip.plipcmd: please cite: Adasme,M. et al. PLIP 2021: expanding the scope of the protein-ligand interaction profiler to DNA and RNA. Nucl. Acids Res. (05 May 2021), gkab294. doi: 10.1093/nar/gkab294
2025-09-06 02:23:36,683 [INFO] [plipcmd.py:49] plip.plipcmd: starting analysis of ENA-Site1_PLIP.pdb
2025-09-06 02:23:37,473 [INFO] [plipcmd.py:165] plip.plipcmd: finished analysis, find the result files in Results/PLIP/HOLO/
2025-09-06 02:23:37,876 [INFO] [plipcmd.py:124] plip.plipcmd: Protein-Ligand Interaction Profiler (PLIP) 2.3.1
2025-09-06 02:23:37,876 [INFO] [plipcmd.py:125] plip.plipcmd: brought to you by: PharmAI GmbH (2020-2021) - www.pharm.ai - hello@pharm.ai
2025-09-06 02:23:37,876 [INF

In [1]:
!pdb_selchain -A HSA_Target/1AO6/1ao6_fixed.pdb > HSA_Target/1AO6/1ao6_fixedA.pdb
!pdb_selchain -A HSA_Target/2BXD/2bxd_fixed.pdb > HSA_Target/2BXD/2bxd_fixedA.pdb
!pdb_selchain -A HSA_Target/2BXG/2bxg_fixed.pdb > HSA_Target/2BXG/2bxg_fixedA.pdb

In [5]:
!plip -f Results/Site1/HOLO/SUC/SUC-Site1_PLIP.pdb -t -x -o Results/PLIP/HOLO --name SUC

2025-09-16 22:11:10,197 [INFO] [plipcmd.py:124] plip.plipcmd: Protein-Ligand Interaction Profiler (PLIP) 2.3.1
2025-09-16 22:11:10,197 [INFO] [plipcmd.py:125] plip.plipcmd: brought to you by: PharmAI GmbH (2020-2021) - www.pharm.ai - hello@pharm.ai
2025-09-16 22:11:10,197 [INFO] [plipcmd.py:126] plip.plipcmd: please cite: Adasme,M. et al. PLIP 2021: expanding the scope of the protein-ligand interaction profiler to DNA and RNA. Nucl. Acids Res. (05 May 2021), gkab294. doi: 10.1093/nar/gkab294
2025-09-16 22:11:10,200 [INFO] [plipcmd.py:49] plip.plipcmd: starting analysis of SUC-Site1_PLIP.pdb
2025-09-16 22:11:10,822 [INFO] [plipcmd.py:165] plip.plipcmd: finished analysis, find the result files in Results/PLIP/HOLO/
